# NEURIA · 뇌 MRI 종양 분류 모델 학습이 노트북은 **Google Colab** 에서 돌리는 것을 전제로 합니다. 학습이 끝나면`web_model.zip` 이 다운로드되고, 그걸 `C:\neuro\model\` 에 풀면 웹사이트가 바로 쓰게 됩니다.### 시작 전 딱 하나상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 로 바꾸고 저장하세요.CPU로 돌리면 2~3시간, GPU로는 15분쯤 걸립니다.> 분류할 4가지는 웹사이트 코드(`js/engine.js`)의 순서와 **반드시** 같아야 합니다:> `glioma(신경교종) → meningioma(수막종) → pituitary(뇌하수체) → notumor(종양 없음)`

## 0. 환경 확인

In [ ]:
import tensorflow as tf, sysprint('Python     :', sys.version.split()[0])print('TensorFlow :', tf.__version__)gpus = tf.config.list_physical_devices('GPU')print('GPU        :', gpus if gpus else '없음 ← 런타임 유형을 T4 GPU로 바꾸세요')

## 1. 데이터셋 받기**Brain Tumor MRI Dataset** (Masoud Nickparvar, Kaggle) — 약 7,000장, 4클래스, `Training/` + `Testing/` 구조.아래 세 방법 중 **하나만** 성공하면 됩니다.

In [ ]:
import os, pathlib, glob, zipfileDATA = None# ── 방법 A: kagglehub (Kaggle 로그인 창이 뜹니다) ─────────────────try:    !pip -q install kagglehub    import kagglehub    DATA = pathlib.Path(kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset"))    print('A 성공:', DATA)except Exception as e:    print('A 실패:', e)# ── 방법 B: kaggle.json 업로드 ───────────────────────────────────# Kaggle → 내 프로필 → Settings → API → Create New Token 으로 받은 kaggle.json 을 업로드if DATA is None:    try:        from google.colab import files        print('kaggle.json 파일을 선택하세요')        files.upload()        os.makedirs('/root/.kaggle', exist_ok=True)        os.replace('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)        !pip -q install kaggle        !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content/dl --unzip        DATA = pathlib.Path('/content/dl')        print('B 성공:', DATA)    except Exception as e:        print('B 실패:', e)# ── 방법 C: 직접 zip 업로드 (Training/ Testing/ 폴더가 들어있는 zip) ──if DATA is None:    from google.colab import files    up = files.upload()    name = list(up)[0]    with zipfile.ZipFile(name) as z: z.extractall('/content/dl')    DATA = pathlib.Path('/content/dl')    print('C 성공:', DATA)# Training / Testing 폴더 위치 자동 탐색def find(nm):    hits = [pathlib.Path(p) for p in glob.glob(str(DATA / '**' / nm), recursive=True)]    hits = [h for h in hits if h.is_dir()]    if not hits: raise SystemExit(f'{nm} 폴더를 찾지 못했습니다. 압축 구조를 확인하세요.')    return sorted(hits, key=lambda p: len(p.parts))[0]TRAIN_DIR, TEST_DIR = find('Training'), find('Testing')print('\nTraining:', TRAIN_DIR)print('Testing :', TEST_DIR)

In [ ]:
# 클래스별 장수 확인 — 숫자가 0이면 폴더 이름이 다른 것입니다CLASS_NAMES = ['glioma', 'meningioma', 'pituitary', 'notumor']   # ← engine.js 순서와 동일KO          = ['신경교종', '수막종', '뇌하수체 종양', '종양 없음']for split, d in [('Training', TRAIN_DIR), ('Testing', TEST_DIR)]:    print(split)    for c in CLASS_NAMES:        n = len(list((d / c).glob('*')))        print(f'  {c:12s} {n:5d} 장' + ('   ← 비어 있음!' if n == 0 else ''))

## 2. 데이터 파이프라인`Training` 을 80/20 으로 나눠 학습·검증에 쓰고, `Testing` 은 **마지막에 한 번만** 쓰는 성능 측정용으로 남겨둡니다.(검증 데이터로 성능을 보고하면 실제보다 좋게 나옵니다.)

In [ ]:
IMG, BATCH, SEED = 224, 32, 1107common = dict(label_mode='int', class_names=CLASS_NAMES,              image_size=(IMG, IMG), batch_size=BATCH, seed=SEED)train_ds = tf.keras.utils.image_dataset_from_directory(    TRAIN_DIR, validation_split=0.2, subset='training',   shuffle=True, **common)val_ds   = tf.keras.utils.image_dataset_from_directory(    TRAIN_DIR, validation_split=0.2, subset='validation', shuffle=True, **common)test_ds  = tf.keras.utils.image_dataset_from_directory(    TEST_DIR, shuffle=False, **common)print('클래스 순서:', train_ds.class_names)assert train_ds.class_names == CLASS_NAMES, '클래스 순서가 어긋났습니다 — 중단하세요'AUTO = tf.data.AUTOTUNEtrain_ds = train_ds.cache().prefetch(AUTO)val_ds   = val_ds.cache().prefetch(AUTO)test_ds  = test_ds.cache().prefetch(AUTO)

In [ ]:
# 눈으로 한 번 확인 — 라벨이 엉뚱하면 여기서 바로 보입니다import matplotlib.pyplot as pltimgs, labs = next(iter(train_ds))plt.figure(figsize=(10, 5))for i in range(10):    plt.subplot(2, 5, i+1); plt.imshow(imgs[i].numpy().astype('uint8'))    plt.title(KO[labs[i]], fontsize=9); plt.axis('off')plt.tight_layout(); plt.show()

## 3. 모델 만들기ImageNet으로 미리 학습된 **MobileNetV2** 위에 분류기만 새로 붙입니다(전이학습).MRI 7천 장으로 처음부터 학습하는 것보다 훨씬 정확하고 빠릅니다.전처리(`Rescaling`)를 **모델 안에** 넣는 게 중요합니다. 그래야 브라우저에서는픽셀값 0~255를 그냥 던져 넣으면 되고, 전처리 수식을 자바스크립트로 또 옮겨 적다 틀릴 일이 없습니다.

In [ ]:
from tensorflow.keras import layers, Model, Inputaug = tf.keras.Sequential([    layers.RandomFlip('horizontal'),    layers.RandomRotation(0.06),    layers.RandomZoom(0.10),    layers.RandomTranslation(0.05, 0.05),    layers.RandomContrast(0.10),], name='augment')base = tf.keras.applications.MobileNetV2(    input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')base.trainable = Falseinputs = Input((IMG, IMG, 3), name='image')          # 0~255 원본 픽셀x = aug(inputs)                                       # 학습 때만 동작x = layers.Rescaling(1/127.5, offset=-1)(x)           # 전처리를 모델에 내장x = base(x, training=False)x = layers.GlobalAveragePooling2D()(x)x = layers.Dropout(0.3)(x)outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', name='probs')(x)model = Model(inputs, outputs, name='neuria_tumor_cls')model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),              loss='sparse_categorical_crossentropy', metrics=['accuracy'])model.summary()

## 4. 1차 학습 — 분류기만MobileNetV2 본체는 얼려두고 맨 위 분류기만 빠르게 맞춥니다. 2~3분.

In [ ]:
cb = [    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),]h1 = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=cb)

## 5. 2차 학습 — 미세조정 (fine-tuning)MobileNetV2 뒷부분을 풀고 **아주 작은 학습률**로 조금씩 움직입니다.학습률을 크게 주면 ImageNet에서 배운 걸 다 잊어버려서 성능이 떨어집니다. 10~12분.

In [ ]:
base.trainable = Truefor l in base.layers[:100]:                 # 앞쪽 100개 층은 그대로 둡니다    l.trainable = Falsemodel.compile(optimizer=tf.keras.optimizers.Adam(1e-5),              loss='sparse_categorical_crossentropy', metrics=['accuracy'])h2 = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb)

In [ ]:
# 학습 곡선 — val 이 train 을 한참 밑돌면 과적합입니다acc  = h1.history['accuracy']      + h2.history['accuracy']vacc = h1.history['val_accuracy']  + h2.history['val_accuracy']plt.figure(figsize=(6,4))plt.plot(acc, label='train'); plt.plot(vacc, label='validation')plt.axvline(len(h1.history['accuracy'])-0.5, ls='--', c='gray')plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.grid(alpha=.3); plt.show()

## 6. 성능 측정 — 여기 숫자가 진짜 성적입니다`Testing` 폴더는 학습에 한 번도 쓰이지 않았습니다. **보고서에 쓸 숫자는 이 칸에서 나온 것만** 쓰세요.특히 혼동행렬에서 **`notumor` 를 종양으로, 종양을 `notumor` 로** 잘못 본 칸을 꼭 보세요 — 의료에서는 이 실수가 가장 위험합니다.

In [ ]:
import numpy as npfrom sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplayloss, acc = model.evaluate(test_ds, verbose=0)print(f'테스트 정확도: {acc*100:.2f}%   (loss {loss:.4f})\n')y_true = np.concatenate([y.numpy() for _, y in test_ds])y_prob = model.predict(test_ds, verbose=0)y_pred = y_prob.argmax(1)print(classification_report(y_true, y_pred, target_names=KO, digits=3, zero_division=0))fig, ax = plt.subplots(figsize=(5.5, 5))ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False)plt.xticks(rotation=30); plt.title('혼동행렬 (행=실제, 열=예측)'); plt.tight_layout(); plt.show()# 종양 ↔ 정상 을 뒤집은 경우만 따로tumor = np.isin(y_true, [0,1,2]); nt = CLASS_NAMES.index('notumor')print(f'종양인데 "종양 없음" 으로 본 수 (놓침) : {int(((y_pred==nt) & tumor).sum())} / {int(tumor.sum())}')print(f'정상인데 종양으로 본 수   (헛경보)   : {int(((y_pred!=nt) & ~tumor).sum())} / {int((~tumor).sum())}')

## 7. 저장 + 웹용(TensorFlow.js)으로 변환브라우저는 `.h5` 를 못 읽습니다. `model.json` + `*.bin` 묶음으로 바꿔야 합니다.아래 칸이 끝나면 **`web_model.zip` 이 자동으로 다운로드**됩니다.

In [ ]:
import json, shutil, subprocess# 원본 보관용 (나중에 재학습·재변환할 때 필요)model.save('neuria_tumor_cls.keras')# SavedModel 로 내보내기 (Keras 3 / 2 둘 다 대응)shutil.rmtree('saved_model', ignore_errors=True)try:    model.export('saved_model')               # Keras 3except AttributeError:    model.save('saved_model', save_format='tf')   # Keras 2print('SavedModel 완료')!pip -q install tensorflowjsshutil.rmtree('web_model', ignore_errors=True)r = subprocess.run(['tensorflowjs_converter',                    '--input_format=tf_saved_model',                    '--output_format=tfjs_graph_model',                    'saved_model', 'web_model'], capture_output=True, text=True)print(r.stdout[-2000:]); print(r.stderr[-2000:])assert os.path.exists('web_model/model.json'), '변환 실패 — 위 메시지를 확인하세요'# 클래스 순서를 같이 넣어둡니다 (나중에 헷갈리지 않도록)json.dump({'classes': CLASS_NAMES, 'ko': KO, 'input_size': IMG,           'preprocessing': 'built-in (0~255 raw RGB in)',           'test_accuracy': round(float(acc), 4)},          open('web_model/labels.json', 'w'), ensure_ascii=False, indent=2)sz = sum(os.path.getsize(os.path.join('web_model', f)) for f in os.listdir('web_model'))print(f'\nweb_model 크기: {sz/1024/1024:.1f} MB')print(os.listdir('web_model'))shutil.make_archive('web_model', 'zip', 'web_model')from google.colab import filesfiles.download('web_model.zip')files.download('neuria_tumor_cls.keras')

## 8. 웹사이트에 붙이기1. 받은 `web_model.zip` 을 풀어서 내용물을 `C:\neuro\model\` 안에 넣습니다   → `C:\neuro\model\model.json`, `group1-shard1of*.bin`, `labels.json`2. TensorFlow.js 라이브러리를 `C:\neuro\js\vendor\tf.min.js` 로 저장합니다   (웹사이트 CSP가 외부 주소를 막아두었기 때문에 **CDN 링크가 아니라 파일로** 받아야 합니다)3. `index.html` 의 스크립트 목록에서 `engine.js` **앞에** 한 줄 추가:   `<script src="js/vendor/tf.min.js"></script>`4. `js/engine.js` 맨 아래에 `train/README.md` 에 적어둔 `modelAnalyze` 블록을 붙여넣습니다자세한 코드는 같은 폴더의 **`train/README.md`** 에 그대로 있습니다.---### 성능이 안 나올 때| 증상 | 손볼 곳 ||---|---|| train 정확도도 낮다 (< 90%) | 1차 학습 epoch 늘리기, 라벨 폴더 확인 || train 높고 val 낮다 (과적합) | `Dropout(0.3)` → `0.5`, 증강 강도 올리기 || 미세조정 후 더 나빠졌다 | 학습률 `1e-5` → `5e-6`, 푸는 층 `[:100]` → `[:130]` || 특정 클래스만 못 맞힌다 | 그 클래스 장수 확인 → `class_weight` 적용 |